# Leakage-Safe CNN-Temporal Soil Moisture Prediction

This notebook implements a deep learning pipeline for soil moisture prediction using a CNN combined with a temporal head (GRU or Transformer). It features absolute "leakage-safe" data splitting and sequences historical satellite imagery.

In [ ]:
try:
    import rasterio
except ImportError:
    print("Installing rasterio...")
    %pip install rasterio

import os
import csv
import math
import random
import copy
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Tuple
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB = True
except ImportError:
    COLAB = False

## 1. Configuration

Update the paths below to match your environment. If you are using Google Drive, ensure the path starts with `/content/drive/MyDrive/...`.

In [ ]:
# --- PATH CONFIGURATION ---
if COLAB:
    # Example relative path in Google Drive
    BASE_DIR = Path('/content/drive/MyDrive/soil_moisture_data')
else:
    BASE_DIR = Path('./')

CSV_PATH = BASE_DIR / "Combined_Area1_Area2_Area3_Area5_Area6_with_tif.csv"
IMAGE_DIR = BASE_DIR / "All_3_areas_cropped"

# --- HYPERPARAMETERS ---
SEQ_LEN = 5
STRIDE = 1
BATCH_SIZE = 32
EPOCHS = 60
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.2
PATIENCE = 10
TEMPORAL_HEAD = "gru" # "gru" or "transformer"
RANDOM_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Data Utilities

These functions handle record loading, temporal splitting (ensuring no leakage between train/val/test), and sequence creation.

In [ ]:
@dataclass(frozen=True)
class Record:
    row_id: int
    area: str
    date_time: datetime
    image_name: str
    target: float

@dataclass(frozen=True)
class SequenceSample:
    split_name: str
    area: str
    records: Tuple[Record, ...]
    image_names: Tuple[str, ...]
    delta_days: Tuple[float, ...]
    cumulative_days: Tuple[float, ...]
    day_of_year_sin: Tuple[float, ...]
    day_of_year_cos: Tuple[float, ...]
    target: float

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def load_records(csv_path: Path, image_dir: Path) -> Tuple[List[Record], Dict[str, Any]]:
    records: List[Record] = []
    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row_idx, row in enumerate(reader):
            image_name = row["Image_name"].strip()
            if not (image_dir / image_name).exists():
                continue
            target = float(row["soil_moisture_depth_0.050000"])
            records.append(Record(
                row_id=row_idx,
                area=row["Area"].strip(),
                date_time=datetime.fromisoformat(row["date_time"].strip()),
                image_name=image_name,
                target=target,
            ))
    return records, {"count": len(records)}

def group_records_by_area(records: List[Record]) -> Dict[str, List[Record]]:
    grouped = defaultdict(list)
    for record in records:
        grouped[record.area].append(record)
    for area_records in grouped.values():
        area_records.sort(key=lambda x: x.date_time)
    return dict(sorted(grouped.items()))

def build_splits(records: List[Record], val_fraction=0.2, test_fraction=0.2, gap_samples=1) -> Dict[str, List[Record]]:
    grouped = group_records_by_area(records)
    splits = {"train": [], "val": [], "test": []}
    for area_records in grouped.values():
        n = len(area_records)
        n_val = max(1, math.ceil(n * val_fraction))
        n_test = max(1, math.ceil(n * test_fraction))
        n_train = n - n_val - n_test - (2 * gap_samples)
        
        train_end = n_train
        val_start = train_end + gap_samples
        val_end = val_start + n_val
        test_start = val_end + gap_samples

        splits["train"].extend(area_records[:train_end])
        splits["val"].extend(area_records[val_start:val_end])
        splits["test"].extend(area_records[test_start:])
    return splits

def build_days_since_previous_lookup(records: List[Record]) -> Dict[int, float]:
    lookup = {}
    grouped = group_records_by_area(records)
    for area_records in grouped.values():
        prev_date = None
        for record in area_records:
            if prev_date is None: lookup[record.row_id] = -1.0
            else: lookup[record.row_id] = float((record.date_time - prev_date).days)
            prev_date = record.date_time
    return lookup

def build_sequences(records, split_name, seq_len, stride, days_lookup) -> List[SequenceSample]:
    sequences = []
    grouped = group_records_by_area(records)
    for area, area_records in grouped.items():
        if len(area_records) < seq_len: continue
        for i in range(0, len(area_records) - seq_len + 1, stride):
            window = area_records[i : i + seq_len]
            delta_days = []
            cumulative_days = []
            doy_sin, doy_cos = [], []
            running_days = 0.0
            for t, record in enumerate(window):
                if t == 0: d = max(0.0, days_lookup.get(record.row_id, 0.0))
                else: d = float((record.date_time - window[t-1].date_time).days)
                running_days += d
                doy = record.date_time.timetuple().tm_yday
                delta_days.append(d)
                cumulative_days.append(running_days)
                doy_sin.append(math.sin(2*math.pi*doy/365.25))
                doy_cos.append(math.cos(2*math.pi*doy/365.25))
            sequences.append(SequenceSample(
                split_name=split_name, area=area, records=tuple(window),
                image_names=tuple(r.image_name for r in window),
                delta_days=tuple(delta_days), cumulative_days=tuple(cumulative_days),
                day_of_year_sin=tuple(doy_sin), day_of_year_cos=tuple(doy_cos),
                target=window[-1].target
            ))
    return sequences

## 3. Preprocessing Stats

Normalization is critical for deep learning. We compute the mean and standard deviation of images, temporal features, and targets using only the training set.

In [ ]:
def import_rasterio():
    import rasterio
    return rasterio

def load_image_lookup(records: List[Record], image_dir: Path) -> Dict[str, np.ndarray]:
    rio = import_rasterio()
    lookup = {}
    img_names = sorted({r.image_name for r in records})
    print(f"Loading {len(img_names)} images...")
    for i, name in enumerate(img_names):
        with rio.open(image_dir / name) as src:
            img = src.read().astype(np.float32)
            img = np.nan_to_num(img, nan=0.0)
            lookup[name] = np.clip(img, 0, None)
        if (i+1) % 500 == 0: print(f"  Loaded {i+1}/{len(img_names)}")
    return lookup

def compute_image_stats(train_records, lookup):
    first = lookup[train_records[0].image_name]
    sums = np.zeros(first.shape[0], dtype=np.float64)
    sq_sums = np.zeros(first.shape[0], dtype=np.float64)
    pixels = 0
    for r in train_records:
        img = lookup[r.image_name]
        flat = img.reshape(img.shape[0], -1).astype(np.float64)
        sums += flat.sum(1)
        sq_sums += (flat**2).sum(1)
        pixels += flat.shape[1]
    means = sums / pixels
    stds = np.sqrt(np.clip((sq_sums / pixels) - means**2, 1e-6, None))
    return means.astype(np.float32), stds.astype(np.float32)

def compute_time_stats(train_seqs):
    rows = []
    for s in train_seqs:
        for i in range(len(s.records)):
            d = float(s.delta_days[i])
            rows.append([d, math.log1p(d), float(s.cumulative_days[i]), s.day_of_year_sin[i], s.day_of_year_cos[i]])
    mat = np.array(rows, dtype=np.float32)
    means = mat.mean(0)
    stds = np.where(mat.std(0) < 1e-6, 1.0, mat.std(0))
    return means, stds

def compute_target_stats(train_seqs):
    targets = np.array([s.target for s in train_seqs])
    return float(targets.mean()), float(max(1e-6, targets.std()))

## 4. Models & Dataset

A CNN encodes the spatial features of each image in the sequence, and a GRU/Transformer processes these embeddings over time.

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, sequences, image_lookup, band_means, band_stds, time_means, time_stds, target_mean, target_std):
        self.sequences = sequences
        self.image_lookup = image_lookup
        self.band_means = torch.tensor(band_means, dtype=torch.float32).view(-1, 1, 1)
        self.band_stds = torch.tensor(band_stds, dtype=torch.float32).view(-1, 1, 1)
        self.time_means = torch.tensor(time_means, dtype=torch.float32)
        self.time_stds = torch.tensor(time_stds, dtype=torch.float32)
        self.target_mean = float(target_mean)
        self.target_std = float(target_std)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        sequence = self.sequences[index]
        images = []
        for image_name in sequence.image_names:
            image = torch.from_numpy(self.image_lookup[image_name]).float()
            image = (image - self.band_means) / self.band_stds
            images.append(image)
        image_tensor = torch.stack(images, dim=0)

        time_rows = []
        for step in range(len(sequence.records)):
            delta_days = float(sequence.delta_days[step])
            time_rows.append([
                delta_days, math.log1p(delta_days), float(sequence.cumulative_days[step]),
                float(sequence.day_of_year_sin[step]), float(sequence.day_of_year_cos[step])
            ])
        time_tensor = torch.tensor(time_rows, dtype=torch.float32)
        time_tensor = (time_tensor - self.time_means) / self.time_stds

        target_norm = (sequence.target - self.target_mean) / self.target_std
        target_tensor = torch.tensor(target_norm, dtype=torch.float32)
        return image_tensor, time_tensor, target_tensor

class ConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size // 2
        
        self.conv_gates = nn.Conv2d(input_dim + hidden_dim, 2 * hidden_dim, kernel_size, padding=padding)
        self.conv_can = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)
        
    def forward(self, input_tensor, h_cur=None):
        if h_cur is None:
            b, _, h, w = input_tensor.shape
            h_cur = torch.zeros(b, self.hidden_dim, h, w, device=input_tensor.device)
            
        combined = torch.cat([input_tensor, h_cur], dim=1)
        gates = self.conv_gates(combined)
        reset_gate, update_gate = torch.split(gates, self.hidden_dim, dim=1)
        
        reset_gate = torch.sigmoid(reset_gate)
        update_gate = torch.sigmoid(update_gate)
        
        combined_reset = torch.cat([input_tensor, reset_gate * h_cur], dim=1)
        can = torch.tanh(self.conv_can(combined_reset))
        
        h_next = (1 - update_gate) * h_cur + update_gate * can
        return h_next


class CNNTemporalRegressor(nn.Module):
    """Spatio-Temporal ConvGRU Regressor."""
    def __init__(
        self,
        in_c,
        seq_len,
        time_dim,
        embed_dim,  # Ignored (kept for arg compatibility)
        hidden_dim,
        layers,     # Ignored (kept for arg compatibility)
        heads,      # Ignored (kept for arg compatibility)
        dropout,
        head_type,  # Ignored (kept for arg compatibility)
    ):
        super().__init__()
        
        self.conv_gru = ConvGRUCell(
            input_dim=in_c + time_dim,
            hidden_dim=hidden_dim,
            kernel_size=3
        )
        
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(1),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, images, time_features):
        b, s, c, h, w = images.shape
        
        time_expanded = time_features.view(b, s, -1, 1, 1).expand(b, s, -1, h, w)
        fused = torch.cat([images, time_expanded], dim=2)
        
        h_cur = None
        for t in range(s):
            h_cur = self.conv_gru(fused[:, t, :, :, :], h_cur)
            
        return self.head(h_cur).squeeze(-1)


## 5. Training Loop

Standard PyTorch training with early stopping.

In [ ]:
def run_epoch(model, loader, opt, loss_fn, device, clip):
    model.train()
    total = 0.0
    for imgs, times, targets in loader:
        imgs, times, targets = imgs.to(device), times.to(device), targets.to(device)
        opt.zero_grad()
        preds = model(imgs, times)
        loss = loss_fn(preds, targets)
        loss.backward()
        if clip > 0: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        total += loss.item() * imgs.size(0)
    return total / len(loader.dataset)

def predict(model, loader, device, tg_m, tg_s):
    model.eval()
    preds = []
    with torch.no_grad():
        for imgs, times, _ in loader:
            p = model(imgs.to(device), times.to(device)).cpu().numpy()
            preds.extend(p)
    return np.array(preds) * tg_s + tg_m

## 6. Main Execution

This cell runs the entire pipeline.

In [ ]:
set_seed(RANDOM_SEED)
records, meta = load_records(CSV_PATH, IMAGE_DIR)
splits = build_splits(records)
days_lookup = build_days_since_previous_lookup(records)

seqs = { k: build_sequences(v, k, SEQ_LEN, STRIDE, days_lookup) for k, v in splits.items() }
lookup = load_image_lookup(records, IMAGE_DIR)

b_m, b_s = compute_image_stats(splits["train"], lookup)
t_m, t_s = compute_time_stats(seqs["train"])
tg_m, tg_s = compute_target_stats(seqs["train"])

train_ds = SequenceDataset(seqs["train"], lookup, b_m, b_s, t_m, t_s, tg_m, tg_s)
val_ds = SequenceDataset(seqs["val"], lookup, b_m, b_s, t_m, t_s, tg_m, tg_s)
test_ds = SequenceDataset(seqs["test"], lookup, b_m, b_s, t_m, t_s, tg_m, tg_s)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

model = CNNTemporalRegressor(
    in_c=int(b_m.shape[0]), seq_len=SEQ_LEN, time_dim=len(t_m),
    embed_dim=128, hidden_dim=128, layers=2, heads=4, dropout=DROPOUT, head_type=TEMPORAL_HEAD
).to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
loss_fn = nn.MSELoss()

best_rmse = float('inf')
best_state = None
history = []
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    t_loss = run_epoch(model, train_loader, opt, loss_fn, DEVICE, 1.0)
    v_preds = predict(model, val_loader, DEVICE, tg_m, tg_s)
    v_targets = np.array([s.target for s in seqs["val"]])
    v_rmse = np.sqrt(mean_squared_error(v_targets, v_preds))
    sch.step(v_rmse)
    
    history.append({"epoch": epoch, "train_loss": t_loss, "val_rmse": v_rmse})
    print(f"Epoch {epoch:02d} | Train Loss: {t_loss:.5f} | Val RMSE: {v_rmse:.5f}")

    if v_rmse < best_rmse:
        best_rmse = v_rmse
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping!"); break

model.load_state_dict(best_state)

## 7. Results Visualization

In [ ]:
epochs = [h['epoch'] for h in history]
t_losses = [h['train_loss'] for h in history]
v_rmses = [h['val_rmse'] for h in history]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, t_losses, label='Train Loss')
plt.title('Training Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, v_rmses, label='Val RMSE', color='orange')
plt.title('Validation RMSE')
plt.legend()
plt.show()

test_preds = predict(model, test_loader, DEVICE, tg_m, tg_s)
test_targets = np.array([s.target for s in seqs["test"]])

plt.figure(figsize=(6, 6))
plt.scatter(test_targets, test_preds, alpha=0.5)
plt.plot([test_targets.min(), test_targets.max()], [test_targets.min(), test_targets.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Test Set: Actual vs Predicted')
plt.grid(True)
plt.show()

mae = mean_absolute_error(test_targets, test_preds)
r2 = r2_score(test_targets, test_preds)
print(f"Test Results -> RMSE: {best_rmse:.5f}, MAE: {mae:.5f}, R2: {r2:.5f}")